# Clean all conference data
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [ ]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.3/486.3 kB 28.5 MB/s eta 0:00:00
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,228 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40

## Cleaning 'ISI'

In [ ]:
# Load the Excel file with multiple sheets
file_path = '/content/drive/My Drive/ISI/ISIclean/ISI2021.xlsx'
xls = pd.ExcelFile(file_path)

In [ ]:
from google.colab import files

output_file = "ISI2021_updated.xlsx"


# Load all sheets into a dictionary of DataFrames
xls = pd.ExcelFile(file_path)
sheets = {sheet_name: xls.parse(sheet_name) for sheet_name in xls.sheet_names}

# Select the 'Authorwise 2018' sheet for processing
authorwise_df = sheets['Authorwise (2021)']

# Add new columns with specified values
authorwise_df['conf'] = 'ISI ACEGD'
authorwise_df['conf_year'] = '2021'

# Rename columns
authorwise_df.rename(columns={
    'Title of Paper': 'paper_title',
    'Topic': 'theme_isi',
    'Number of Authors': 'authors_total'
}, inplace=True)

# Create new columns for male and female authors
authorwise_df['authors_total_male'] = 0
authorwise_df['authors_total_female'] = 0

# Count male and female authors per paper_title
def count_authors_by_gender(group):
    male_count = (group['Gender'] == 'Male').sum()
    female_count = (group['Gender'] == 'Female').sum()
    group['authors_total_male'] = male_count
    group['authors_total_female'] = female_count
    return group

authorwise_df = authorwise_df.groupby('paper_title').apply(count_authors_by_gender)

# Save the updated DataFrame back to the original dictionary of sheets
sheets['Authorwise (2021)'] = authorwise_df

# Write all sheets back to an Excel file
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# Save the file to Google Drive
files.download(output_file)

print(f"Updated file saved as {output_file} and downloaded to your system.")

<ipython-input-4-fdfc90b3669a>:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  authorwise_df = authorwise_df.groupby('paper_title').apply(count_authors_by_gender)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Updated file saved as ISI2021_updated.xlsx and downloaded to your system.


In [ ]:
import pandas as pd
import re

# Load the spreadsheet
file_path = '/content/drive/My Drive/ISI/ISIclean/ISI2021_updated.xlsx'
sheet_name = 'Authorwise (2021)'

# Load the Authorwise 2018 sheet
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Remove characters and text in between '():' in theme_isi column
#df['theme_isi'] = df['theme_isi'].apply(lambda x: re.sub(r'\([a-zA-Z]\):', '', x))

# Create new columns with NaN values
new_columns = [
    'presenter_autho', 'author_name_1', 'author_gender_1', 'author_qual_1', 'author_qual_1_o',
    'department_1', 'author_insti_1', 'author_insti_country1', 'phd_comp_1', 'phd_subject_1',
    'phd_insti_1', 'phd_inscountry_1', 'phd_yr_1', 'masters_comp_1', 'masters_insti_1',
    'masters_country1', 'masters_yr_1', 'underg_subject_1', 'underg_insti_1', 'underg_country_1',
    'underg_yr_1', 'email1_1', 'email2_1', 'author_name_2', 'author_gender_2', 'author_qual_2',
    'author_qual_2_o', 'department_2', 'author_insti_2', 'author_insti_country2', 'phd_comp_2',
    'phd_subject_2', 'phd_insti_2', 'phd_inscountry_2', 'phd_yr_2', 'masters_comp_2',
    'masters_insti_2', 'masters_country2', 'masters_yr_2', 'underg_subject_2', 'underg_insti_2',
    'underg_country_2', 'underg_yr_2', 'email1_2', 'email2_2', 'author_name_3', 'author_gender_3',
    'author_qual_3', 'author_qual_3_o', 'department_3', 'author_insti_3', 'author_insti_country3',
    'phd_comp_3', 'phd_subject_3', 'phd_insti_3', 'phd_inscountry_3', 'phd_yr_3', 'masters_comp_3',
    'masters_insti_3', 'masters_country3', 'masters_yr_3', 'underg_subject_3', 'underg_insti_3',
    'underg_country_3', 'underg_yr_3', 'email1_3', 'email2_3', 'author_name_4', 'author_gender_4',
    'author_qual_4', 'author_qual_4_o', 'department_4', 'author_insti_4', 'author_insti_country4',
    'phd_comp_4', 'phd_subject_4', 'phd_insti_4', 'phd_inscountry_4', 'phd_yr_4', 'masters_comp_4',
    'masters_insti_4', 'masters_country4', 'masters_yr_4', 'underg_subject_4', 'underg_insti_4',
    'underg_country_4', 'underg_yr_4', 'email1_4', 'email2_4', 'author_name_5', 'author_gender_5',
    'author_qual_5', 'author_qual_5_o', 'department_5', 'author_insti_5', 'author_insti_country5',
    'phd_comp_5', 'phd_subject_5', 'phd_insti_5', 'phd_inscountry_5', 'phd_yr_5', 'masters_comp_5',
    'masters_insti_5', 'masters_country5', 'masters_yr_5', 'underg_subject_5', 'underg_insti_5',
    'underg_country_5', 'underg_yr_5', 'email1_5', 'email2_5', 'author_name_6', 'author_gender_6',
    'author_qual_6', 'author_qual_6_o', 'department_6', 'author_insti_6', 'author_insti_country6',
    'phd_comp_6', 'phd_subject_6', 'phd_insti_6', 'phd_inscountry_6', 'phd_yr_6', 'masters_comp_6',
    'masters_insti_6', 'masters_country6', 'masters_yr_6', 'underg_subject_6', 'underg_insti_6',
    'underg_country_6', 'underg_yr_6', 'email1_6', 'email2_6', 'author_name_7', 'author_gender_7',
    'author_qual_7', 'author_qual_7_o', 'department_7', 'author_insti_7', 'author_insti_country7',
    'phd_comp_7', 'phd_subject_7', 'phd_insti_7', 'phd_inscountry_7', 'phd_yr_7', 'masters_comp_7',
    'masters_insti_7', 'masters_country7', 'masters_yr_7', 'underg_subject_7', 'underg_insti_7',
    'underg_country_7', 'underg_yr_7', 'email1_7', 'Email2_7'
]

# Use pd.concat to add all new columns at once to avoid fragmentation
df = pd.concat([df, pd.DataFrame(columns=new_columns)], axis=1)

# Prepare a new DataFrame to hold the transformed data
transformed_data = []

# Fill the new DataFrame with data
unique_papers = df['paper_title'].unique()
for paper in unique_papers:
    paper_data = df[df['paper_title'] == paper]
    authors = paper_data['Author'].unique()
    new_row = paper_data.iloc[0].to_dict()

    for j, author in enumerate(authors):
        author_idx = j + 1
        # Check if the filtered DataFrame is empty before accessing .iloc[0]
        filtered_author_data = paper_data[paper_data['Author'] == author]
        if not filtered_author_data.empty:
            author_info = filtered_author_data.iloc[0]

        new_row[f'author_name_{author_idx}'] = author
        new_row[f'author_insti_{author_idx}'] = author_info['Institute/Affiliation']
        new_row[f'author_insti_country{author_idx}'] = author_info['Affiliation Country']
        new_row[f'author_gender_{author_idx}'] = author_info['Gender']
        #new_row[f'author_qual_{author_idx}'] = author_info['Author Qualification']
        #new_row[f'department_{author_idx}'] = author_info['Department']
        #new_row[f'phd_comp_{author_idx}'] = author_info['PhD Completed']
        #new_row[f'phd_subject_{author_idx}'] = author_info['PhD Subject']
        #new_row[f'phd_insti_{author_idx}'] = author_info['PhD Institute']
        #new_row[f'phd_inscountry_{author_idx}'] = author_info['PhD Institute Country']
        #new_row[f'phd_yr_{author_idx}'] = author_info['PhD Year']
        #new_row[f'masters_comp_{author_idx}'] = author_info['Masters Completed']
        #new_row[f'masters_insti_{author_idx}'] = author_info['Masters Institute']
        #new_row[f'masters_country_{author_idx}'] = author_info['Masters Country']
        #new_row[f'masters_yr_{author_idx}'] = author_info['Masters Year']
        #new_row[f'underg_subject_{author_idx}'] = author_info['Undergraduate Subject']
        #new_row[f'underg_insti_{author_idx}'] = author_info['Undergraduate Institute']
        #new_row[f'underg_country_{author_idx}'] = author_info['Undergraduate Country']
        #new_row[f'underg_yr_{author_idx}'] = author_info['Undergraduate Year']
        #new_row[f'email1_{author_idx}'] = author_info['Email 1']
        #new_row[f'email2_{author_idx}'] = author_info['Email 2']

    transformed_data.append(new_row)

transformed_df = pd.DataFrame(transformed_data)

# Save the updated dataframe back to Excel
updated_file_path = '/content/drive/My Drive/ISI/ISIclean/ISI2021_updated1.xlsx'
with pd.ExcelWriter(updated_file_path, engine='openpyxl', mode='w') as writer:
    transformed_df.to_excel(writer, sheet_name=sheet_name, index=False)

# Confirm the file has been updated
print(f"Updated file saved to {updated_file_path}")


Updated file saved to /content/drive/My Drive/ISI/ISIclean/ISI2021_updated1.xlsx


In [ ]:
import pandas as pd

# Load ISI2018_updated file
isi2018_file_path = '/content/drive/My Drive/ISI/ISIclean/ISI2021.xlsx'
isi2018_sheet_name = 'Authorwise (2021)'
df_isi2018 = pd.read_excel(isi2018_file_path, sheet_name=isi2018_sheet_name)

# Load ISI2019_updated file
isi2019_file_path = '/content/drive/My Drive/ISI/ISIclean/ISI2021_updated1.xlsx'
isi2019_sheet_name = 'Authorwise (2021)'
df_isi2019 = pd.read_excel(isi2019_file_path, sheet_name=isi2019_sheet_name)

# Check the number of unique paper_title in ISI2018
unique_papers_2018 = df_isi2018['Title of Paper'].nunique()
print(f"Number of unique paper_title in ISI2023: {unique_papers_2018}")

# Check the number of unique paper_title in ISI2019
unique_papers_2018up = df_isi2019['paper_title'].nunique()
print(f"Number of unique paper_title in ISI2023updated: {unique_papers_2018up}")


Number of unique paper_title in ISI2023: 195
Number of unique paper_title in ISI2023updated: 195


In [ ]:
import pandas as pd


# Load ISI2018_updated1 file
file_path = '/content/drive/My Drive/ISI/ISIclean/ISI2021_updated1.xlsx'
sheet_name = 'Authorwise (2021)'

# Load the entire workbook
wb = pd.ExcelFile(file_path)

# Load the Authorwise 2018 sheet
df = wb.parse(sheet_name)

# Drop the 'Session' column
if 'Session' in df.columns:
    df.drop(columns=['Session'], inplace=True)

# Define the desired order of columns
desired_columns = [
    'conf', 'conf_year', 'paper_title', 'theme_isi', 'authors_total', 'authors_total_male',
    'authors_total_female', 'presenter_author_no', 'author_name_1', 'author_gender_1',
    'author_qual_1', 'author_qual_1_os', 'department_1', 'author_insti_1', 'author_insti_country_1',
    'phd_comp_1', 'phd_subject_1', 'phd_insti_1', 'phd_inscountry_1', 'phd_yr_1', 'masters_comp_1',
    'masters_comp_1', 'masters_insti_1', 'masters_country_1', 'masters_yr_1', 'underg_subject_1',
    'underg_insti_1', 'underg_country_1', 'underg_yr_1', 'email1_1', 'email2_1', 'author_name_2',
    'author_gender_2', 'author_qual_2', 'author_qual_2_os', 'department_2', 'author_insti_2',
    'author_insti_country_2', 'phd_comp_2', 'phd_subject_2', 'phd_insti_2', 'phd_inscountry_2',
    'phd_yr_2', 'masters_comp_2', 'masters_comp_2', 'masters_insti_2', 'masters_country_2',
    'masters_yr_2', 'underg_subject_2', 'underg_insti_2', 'underg_country_2', 'underg_yr_2',
    'email1_2', 'email2_2', 'author_name_3', 'author_gender_3', 'author_qual_3', 'author_qual_3_os',
    'department_3', 'author_insti_3', 'author_insti_country_3', 'phd_comp_3', 'phd_subject_3',
    'phd_insti_3', 'phd_inscountry_3', 'phd_yr_3', 'masters_comp_3', 'masters_comp_3',
    'masters_insti_3', 'masters_country_3', 'masters_yr_3', 'underg_subject_3', 'underg_insti_3',
    'underg_country_3', 'underg_yr_3', 'email1_3', 'email2_3', 'author_name_4', 'author_gender_4',
    'author_qual_4', 'author_qual_4_os', 'department_4', 'author_insti_4', 'author_insti_country_4',
    'phd_comp_4', 'phd_subject_4', 'phd_insti_4', 'phd_inscountry_4', 'phd_yr_4', 'masters_comp_4',
    'masters_comp_4', 'masters_insti_4', 'masters_country_4', 'masters_yr_4', 'underg_subject_4',
    'underg_insti_4', 'underg_country_4', 'underg_yr_4', 'email1_4', 'email2_4', 'author_name_5',
    'author_gender_5', 'author_qual_5', 'author_qual_5_os', 'department_5', 'author_insti_5',
    'author_insti_country_5', 'phd_comp_5', 'phd_subject_5', 'phd_insti_5', 'phd_inscountry_5',
    'phd_yr_5', 'masters_comp_5', 'masters_comp_5', 'masters_insti_5', 'masters_country_5',
    'masters_yr_5', 'underg_subject_5', 'underg_insti_5', 'underg_country_5', 'underg_yr_5',
    'email1_5', 'email2_5', 'author_name_6', 'author_gender_6', 'author_qual_6', 'author_qual_6_os',
    'department_6', 'author_insti_6', 'author_insti_country_6', 'phd_comp_6', 'phd_subject_6',
    'phd_insti_6', 'phd_inscountry_6', 'phd_yr_6', 'masters_comp_6', 'masters_comp_6',
    'masters_insti_6', 'masters_country_6', 'masters_yr_6', 'underg_subject_6', 'underg_insti_6',
    'underg_country_6', 'underg_yr_6', 'email1_6', 'email2_6', 'author_name_7', 'author_gender_7',
    'author_qual_7', 'author_qual_7_os', 'department_7', 'author_insti_7', 'author_insti_country_7',
    'phd_comp_7', 'phd_subject_7', 'phd_insti_7', 'phd_inscountry_7', 'phd_yr_7', 'masters_comp_7',
    'masters_comp_7', 'masters_insti_7', 'masters_country_7', 'masters_yr_7', 'underg_subject_7',
    'underg_insti_7', 'underg_country_7', 'underg_yr_7', 'email1_7', 'email2_7'
]

# Filter the desired columns to keep only those that exist in the DataFrame
desired_columns = [col for col in desired_columns if col in df.columns]

# Identify any columns in the dataframe that are not in the desired order list
remaining_columns = [col for col in df.columns if col not in desired_columns]

# Reorder columns
df = df[desired_columns + remaining_columns]

# Save the updated dataframe back to the same file
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a') as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)

# Ensure other sheets remain unchanged by writing them back as well
for sheet in wb.sheet_names:
    if sheet != sheet_name:
        wb.parse(sheet).to_excel(writer, sheet_name=sheet, index=False)

# Save the file
writer.save()


ValueError: Sheet 'Authorwise (2021)' already exists and if_sheet_exists is set to 'error'.

### Merging

In [ ]:
import pandas as pd

# File paths and expected sheet names
file_paths = {
    'ISI2018_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2018_updated1.xlsx',
    'ISI2019_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2019_updated1.xlsx',
    #'ISI2020_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2020_updated1.xlsx',
    'ISI2021_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2021_updated1.xlsx',
    'ISI2022_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2022_updated1.xlsx',
    'ISI2023_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2023_updated1.xlsx'
}

final_file_path = '/content/drive/My Drive/ISI/ISIclean/ISIconferences2018-2023.xlsx'

# Load all files and check columns
data_frames = []
reference_columns = None
mismatch_files = []

for file_name, file_path in file_paths.items():
    df = pd.read_excel(file_path)
    data_frames.append(df)
    if reference_columns is None:
        reference_columns = set(df.columns)
    else:
        current_columns = set(df.columns)
        if current_columns != reference_columns:
            mismatch_files.append((file_name, list(reference_columns.symmetric_difference(current_columns))))

if mismatch_files:
    for file, mismatched_columns in mismatch_files:
        print(f"Columns not matching in file {file}: {mismatched_columns}")
else:
    print("All files have matching columns.")

# Concatenate all data frames
combined_df = pd.concat(data_frames, ignore_index=True)

# Sort by 'conf_year'
combined_df.sort_values(by='conf_year', inplace=True)

# Save the concatenated data to a new Excel file with separate sheets
with pd.ExcelWriter(final_file_path, engine='openpyxl') as writer:
    for file_name, file_path in file_paths.items():
        df = pd.read_excel(file_path)
        sheet_name = file_name.replace('.xlsx', '').replace('updated1', '').strip('_')
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    combined_df.to_excel(writer, sheet_name='Paper Presenters', index=False)

# Confirm the file has been updated
print(f"Concatenated file saved to {final_file_path}")


Columns not matching in file ISI2019_updated1.xlsx: ['author_insti_97', 'author_insti_country15', 'author_insti_60', 'author_insti_country97', 'author_gender_28', 'author_name_63', 'author_name_58', 'author_insti_95', 'author_name_16', 'author_insti_23', 'author_gender_70', 'author_name_67', 'author_insti_country32', 'author_insti_49', 'author_insti_country44', 'author_name_36', 'author_gender_46', 'author_gender_64', 'author_gender_68', 'author_name_56', 'author_name_33', 'author_name_69', 'author_insti_22', 'author_gender_34', 'author_insti_53', 'author_insti_20', 'author_gender_77', 'author_insti_country33', 'author_insti_country98', 'author_gender_87', 'author_insti_country43', 'author_insti_18', 'author_insti_73', 'author_gender_25', 'author_gender_16', 'author_insti_country67', 'author_gender_95', 'author_insti_country73', 'author_gender_26', 'author_insti_98', 'author_gender_59', 'author_insti_country92', 'author_insti_country39', 'author_insti_47', 'author_insti_country91', 'au

# Cleaning Ashoka

In [ ]:
# Load the Excel file with multiple sheets
file_path = '/content/drive/My Drive/clean/ashoka/ashoka.xlsx'
xls = pd.ExcelFile(file_path)

In [ ]:
from google.colab import files
sheets = {sheet_name: xls.parse(sheet_name) for sheet_name in xls.sheet_names}

# Process each sheet and save it as a separate Excel file
for sheet_name, df in sheets.items():
    # Add new columns with specified values
    df['conf'] = 'AU AEC'
    df['conf_year'] = sheet_name

    # Rename the column 'Author\'s Gender(Female=1,Male=0)' into 'Gender'
    df.rename(columns={"Author's Gender(Female=1,Male=0)": "Gender"}, inplace=True)

    # Rename columns
    df.rename(columns={
        'Paper Title': 'paper_title',
        'Topic': 'theme_isi',
        'Number of Authors': 'authors_total'
    }, inplace=True)

    # Create new columns for male and female authors
    df['authors_total_male'] = 0
    df['authors_total_female'] = 0

    # Count male and female authors per paper_title
    def count_authors_by_gender(group):
        male_count = (group['Gender'] == 0).sum()
        female_count = (group['Gender'] == 1).sum()
        group['authors_total_male'] = male_count
        group['authors_total_female'] = female_count
        return group

    df = df.groupby('paper_title').apply(count_authors_by_gender)

    # Save the individual sheet as a new Excel file
    output_file = f"{sheet_name}_updated.xlsx"
    df.to_excel(output_file, index=False)

    # Download the file
    files.download(output_file)

print("All sheets have been processed and downloaded separately.")


<ipython-input-6-346fc34eb082>:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('paper_title').apply(count_authors_by_gender)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<ipython-input-6-346fc34eb082>:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('paper_title').apply(count_authors_by_gender)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (12 outputs left out)

<ipython-input-6-346fc34eb082>:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('paper_title').apply(count_authors_by_gender)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

KeyError: 'paper_title'

In [ ]:
from google.colab import files
import pandas as pd
import re

# Define the input and output file paths
input_file_path = '/content/drive/My Drive/clean/ashoka/auconf_updatedd.xlsx'
output_file_path = '/content/drive/My Drive/clean/ashoka/auconf_updated1.xlsx'

# Load all sheets into a dictionary of DataFrames
xls = pd.ExcelFile(input_file_path)
sheets = {sheet_name: xls.parse(sheet_name) for sheet_name in xls.sheet_names}

# Process each sheet from 2017 to 2023
for year in range(2017, 2024):
    sheet_name = str(year)
    if sheet_name in sheets:
        df = sheets[sheet_name]

        # Remove characters and text in between '():' in theme_isi column
        if 'theme_isi' in df.columns:
            df['theme_isi'] = df['theme_isi'].apply(lambda x: re.sub(r'\([a-zA-Z]\):', '', str(x)))

        # Add new columns with NaN values
        new_columns = [
            'presenter_autho', 'author_name_1', 'author_gender_1', 'author_qual_1', 'author_qual_1_o',
            'department_1', 'author_insti_1', 'author_insti_country1', 'phd_comp_1', 'phd_subject_1',
            'phd_insti_1', 'phd_inscountry_1', 'phd_yr_1', 'masters_comp_1', 'masters_insti_1',
            'masters_country1', 'masters_yr_1', 'underg_subject_1', 'underg_insti_1', 'underg_country_1',
            'underg_yr_1', 'email1_1', 'email2_1', 'author_name_2', 'author_gender_2', 'author_qual_2',
            'author_qual_2_o', 'department_2', 'author_insti_2', 'author_insti_country2', 'phd_comp_2',
            'phd_subject_2', 'phd_insti_2', 'phd_inscountry_2', 'phd_yr_2', 'masters_comp_2',
            'masters_insti_2', 'masters_country2', 'masters_yr_2', 'underg_subject_2', 'underg_insti_2',
            'underg_country_2', 'underg_yr_2', 'email1_2', 'email2_2', 'author_name_3', 'author_gender_3',
            'author_qual_3', 'author_qual_3_o', 'department_3', 'author_insti_3', 'author_insti_country3',
            'phd_comp_3', 'phd_subject_3', 'phd_insti_3', 'phd_inscountry_3', 'phd_yr_3', 'masters_comp_3',
            'masters_insti_3', 'masters_country3', 'masters_yr_3', 'underg_subject_3', 'underg_insti_3',
            'underg_country_3', 'underg_yr_3', 'email1_3', 'email2_3', 'author_name_4', 'author_gender_4',
            'author_qual_4', 'author_qual_4_o', 'department_4', 'author_insti_4', 'author_insti_country4',
            'phd_comp_4', 'phd_subject_4', 'phd_insti_4', 'phd_inscountry_4', 'phd_yr_4', 'masters_comp_4',
            'masters_insti_4', 'masters_country4', 'masters_yr_4', 'underg_subject_4', 'underg_insti_4',
            'underg_country_4', 'underg_yr_4', 'email1_4', 'email2_4', 'author_name_5', 'author_gender_5',
            'author_qual_5', 'author_qual_5_o', 'department_5', 'author_insti_5', 'author_insti_country5',
            'phd_comp_5', 'phd_subject_5', 'phd_insti_5', 'phd_inscountry_5', 'phd_yr_5', 'masters_comp_5',
            'masters_insti_5', 'masters_country5', 'masters_yr_5', 'underg_subject_5', 'underg_insti_5',
            'underg_country_5', 'underg_yr_5', 'email1_5', 'email2_5', 'author_name_6', 'author_gender_6',
            'author_qual_6', 'author_qual_6_o', 'department_6', 'author_insti_6', 'author_insti_country6',
            'phd_comp_6', 'phd_subject_6', 'phd_insti_6', 'phd_inscountry_6', 'phd_yr_6', 'masters_comp_6',
            'masters_insti_6', 'masters_country6', 'masters_yr_6', 'underg_subject_6', 'underg_insti_6',
            'underg_country_6', 'underg_yr_6', 'email1_6', 'email2_6', 'author_name_7', 'author_gender_7',
            'author_qual_7', 'author_qual_7_o', 'department_7', 'author_insti_7', 'author_insti_country7',
            'phd_comp_7', 'phd_subject_7', 'phd_insti_7', 'phd_inscountry_7', 'phd_yr_7', 'masters_comp_7',
            'masters_insti_7', 'masters_country7', 'masters_yr_7', 'underg_subject_7', 'underg_insti_7',
            'underg_country_7', 'underg_yr_7', 'email1_7', 'Email2_7'
        ]
        df = pd.concat([df, pd.DataFrame(columns=new_columns)], axis=1)

        # Prepare a new DataFrame to hold the transformed data
        transformed_data = []

        # Fill the new DataFrame with data
        if 'paper_title' in df.columns and 'Author' in df.columns:
            unique_papers = df['paper_title'].unique()
            for paper in unique_papers:
                paper_data = df[df['paper_title'] == paper]
                authors = paper_data['Author'].unique()
                new_row = paper_data.iloc[0].to_dict()

                for j, author in enumerate(authors):
                    author_idx = j + 1
                    # Check if the filtered DataFrame is empty before accessing .iloc[0]
                    filtered_author_data = paper_data[paper_data['Author'] == author]
                    if not filtered_author_data.empty:
                        author_info = filtered_author_data.iloc[0]
                        new_row[f'author_name_{author_idx}'] = author
                        new_row[f'author_insti_{author_idx}'] = author_info.get('Institute/Affiliation', '')
                        new_row[f'author_insti_country{author_idx}'] = author_info.get('Affiliation Country', '')
                        new_row[f'author_gender_{author_idx}'] = author_info.get('Gender', '')

                transformed_data.append(new_row)

        transformed_df = pd.DataFrame(transformed_data)

        # Update the sheet in the dictionary
        sheets[sheet_name] = transformed_df

# Save all sheets back to the Excel file
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# Save the file to Google Drive
files.download(output_file_path)

print(f"Updated file saved as {output_file_path} and downloaded to your system.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Updated file saved as /content/drive/My Drive/clean/ashoka/auconf_updated1.xlsx and downloaded to your system.


In [ ]:
from google.colab import files
import pandas as pd
import os

# Define the input and output directory paths
input_dir = '/content/drive/My Drive/clean/ashoka/'
output_dir = '/content/drive/My Drive/clean/ashoka/processed/'

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# List all Excel files in the input directory
input_files = [f for f in os.listdir(input_dir) if f.endswith('.xlsx')]

# Process each individual Excel file
for input_file in input_files:
    # Define the input and output file paths for the individual sheet
    input_file_path = os.path.join(input_dir, input_file)
    output_file_path = os.path.join(output_dir, f"processed_{input_file}")

    # Load the Excel file to get the sheet names
    xls = pd.ExcelFile(input_file_path)

    # Process each sheet
    for sheet_name in xls.sheet_names:
        # Read the sheet into a DataFrame
        df = pd.read_excel(xls, sheet_name=sheet_name)

        # Remove characters and text in between '():' in theme_isi column if it exists
        if 'theme_isi' in df.columns:
            df['theme_isi'] = df['theme_isi'].apply(lambda x: re.sub(r'\([a-zA-Z]\):', '', str(x)))

        # Add new columns with NaN values
        new_columns = [
            'presenter_autho', 'author_name_1', 'author_gender_1', 'author_qual_1', 'author_qual_1_o',
            'department_1', 'author_insti_1', 'author_insti_country1', 'phd_comp_1', 'phd_subject_1',
            'phd_insti_1', 'phd_inscountry_1', 'phd_yr_1', 'masters_comp_1', 'masters_insti_1',
            'masters_country1', 'masters_yr_1', 'underg_subject_1', 'underg_insti_1', 'underg_country_1',
            'underg_yr_1', 'email1_1', 'email2_1', 'author_name_2', 'author_gender_2', 'author_qual_2',
            'author_qual_2_o', 'department_2', 'author_insti_2', 'author_insti_country2', 'phd_comp_2',
            'phd_subject_2', 'phd_insti_2', 'phd_inscountry_2', 'phd_yr_2', 'masters_comp_2',
            'masters_insti_2', 'masters_country2', 'masters_yr_2', 'underg_subject_2', 'underg_insti_2',
            'underg_country_2', 'underg_yr_2', 'email1_2', 'email2_2', 'author_name_3', 'author_gender_3',
            'author_qual_3', 'author_qual_3_o', 'department_3', 'author_insti_3', 'author_insti_country3',
            'phd_comp_3', 'phd_subject_3', 'phd_insti_3', 'phd_inscountry_3', 'phd_yr_3', 'masters_comp_3',
            'masters_insti_3', 'masters_country3', 'masters_yr_3', 'underg_subject_3', 'underg_insti_3',
            'underg_country_3', 'underg_yr_3', 'email1_3', 'email2_3', 'author_name_4', 'author_gender_4',
            'author_qual_4', 'author_qual_4_o', 'department_4', 'author_insti_4', 'author_insti_country4',
            'phd_comp_4', 'phd_subject_4', 'phd_insti_4', 'phd_inscountry_4', 'phd_yr_4', 'masters_comp_4',
            'masters_insti_4', 'masters_country4', 'masters_yr_4', 'underg_subject_4', 'underg_insti_4',
            'underg_country_4', 'underg_yr_4', 'email1_4', 'email2_4', 'author_name_5', 'author_gender_5',
            'author_qual_5', 'author_qual_5_o', 'department_5', 'author_insti_5', 'author_insti_country5',
            'phd_comp_5', 'phd_subject_5', 'phd_insti_5', 'phd_inscountry_5', 'phd_yr_5', 'masters_comp_5',
            'masters_insti_5', 'masters_country5', 'masters_yr_5', 'underg_subject_5', 'underg_insti_5',
            'underg_country_5', 'underg_yr_5', 'email1_5', 'email2_5', 'author_name_6', 'author_gender_6',
            'author_qual_6', 'author_qual_6_o', 'department_6', 'author_insti_6', 'author_insti_country6',
            'phd_comp_6', 'phd_subject_6', 'phd_insti_6', 'phd_inscountry_6', 'phd_yr_6', 'masters_comp_6',
            'masters_insti_6', 'masters_country6', 'masters_yr_6', 'underg_subject_6', 'underg_insti_6',
            'underg_country_6', 'underg_yr_6', 'email1_6', 'email2_6', 'author_name_7', 'author_gender_7',
            'author_qual_7', 'author_qual_7_o', 'department_7', 'author_insti_7', 'author_insti_country7',
            'phd_comp_7', 'phd_subject_7', 'phd_insti_7', 'phd_inscountry_7', 'phd_yr_7', 'masters_comp_7',
            'masters_insti_7', 'masters_country7', 'masters_yr_7', 'underg_subject_7', 'underg_insti_7',
            'underg_country_7', 'underg_yr_7', 'email1_7', 'Email2_7'
        ]
        df = pd.concat([df, pd.DataFrame(columns=new_columns)], axis=1)

        # Prepare a new DataFrame to hold the transformed data
        transformed_data = []

        # Fill the new DataFrame with data
        if 'paper_title' in df.columns and 'Author' in df.columns:
            unique_papers = df['paper_title'].unique()
            for paper in unique_papers:
                paper_data = df[df['paper_title'] == paper]
                authors = paper_data['Author'].unique()
                new_row = paper_data.iloc[0].to_dict()

                for j, author in enumerate(authors):
                    author_idx = j + 1
                    filtered_author_data = paper_data[paper_data['Author'] == author]
                    if not filtered_author_data.empty:
                        author_info = filtered_author_data.iloc[0]
                        new_row[f'author_name_{author_idx}'] = author
                        new_row[f'author_insti_{author_idx}'] = author_info.get('Institute/Affiliation', '')
                        new_row[f'author_insti_country{author_idx}'] = author_info.get('Affiliation Country', '')
                        new_row[f'author_gender_{author_idx}'] = author_info.get('Gender', '')

                transformed_data.append(new_row)

        transformed_df = pd.DataFrame(transformed_data)

        # Save the transformed DataFrame to the output file, preserving the sheet name
        sheet_output_path = os.path.join(output_dir, f"{sheet_name}_processed_{input_file}")
        transformed_df.to_excel(sheet_output_path, index=False)

        # Download the file
        files.download(sheet_output_path)

print(f"All sheets have been processed and downloaded separately.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (67 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All sheets have been processed and downloaded separately.


Check columns numbers and names

In [ ]:
import pandas as pd
import os

# Define the directory path where processed files are stored
processed_dir = '/content/drive/My Drive/clean/ashoka/process/'

# List all Excel files in the processed directory
processed_files = [f for f in os.listdir(processed_dir) if f.endswith('_processed.xlsx')]

# Check if there are any files
if not processed_files:
    print("No processed files found in the directory.")
else:
    # Initialize a variable to store column names for comparison
    previous_columns = None

    # Loop through each processed file
    for processed_file in processed_files:
        # Define the full file path
        processed_file_path = os.path.join(processed_dir, processed_file)

        # Load the file into a DataFrame
        try:
            df = pd.read_excel(processed_file_path)
        except Exception as e:
            print(f"Failed to load {processed_file}: {e}")
            continue

        # Get the column names for this file
        current_columns = df.columns.tolist()

        # Check if columns are similar to previous file
        if previous_columns is not None:
            if previous_columns != current_columns:
                print(f"Column differences found in {processed_file}:")
                print(f"Columns in previous file: {previous_columns}")
                print(f"Columns in current file: {current_columns}")
                print('-' * 50)

        # Update the previous_columns for comparison with the next file
        previous_columns = current_columns

        # Print out the number of columns in the current file
        print(f"File: {processed_file} has {len(current_columns)} columns.")

    print("Column comparison and file details have been printed.")


File: 2023_processed.xlsx has 169 columns.
Column differences found in 2018_processed.xlsx:
Columns in previous file: ['Session', 'theme_isi', 'paper_title', 'authors_total', 'Author', 'Institute/Affiliation', 'Source for Author Names', 'Gender', 'Source used for determining gender', 'Country', 'conf', 'conf_year', 'authors_total_male', 'authors_total_female', 'presenter_autho', 'author_name_1', 'author_gender_1', 'author_qual_1', 'author_qual_1_o', 'department_1', 'author_insti_1', 'author_insti_country1', 'phd_comp_1', 'phd_subject_1', 'phd_insti_1', 'phd_inscountry_1', 'phd_yr_1', 'masters_comp_1', 'masters_insti_1', 'masters_country1', 'masters_yr_1', 'underg_subject_1', 'underg_insti_1', 'underg_country_1', 'underg_yr_1', 'email1_1', 'email2_1', 'author_name_2', 'author_gender_2', 'author_qual_2', 'author_qual_2_o', 'department_2', 'author_insti_2', 'author_insti_country2', 'phd_comp_2', 'phd_subject_2', 'phd_insti_2', 'phd_inscountry_2', 'phd_yr_2', 'masters_comp_2', 'masters_ins

merge ashoka data

In [ ]:
import pandas as pd
import os

# Define the directory path where processed files are stored
processed_dir = '/content/drive/My Drive/clean/ashoka/process/'

# List all Excel files in the processed directory
processed_files = [f for f in os.listdir(processed_dir) if f.endswith('_processed.xlsx')]

# Initialize an empty list to store DataFrames
dfs = []

# Loop through each processed file
for processed_file in processed_files:
    # Define the full file path
    processed_file_path = os.path.join(processed_dir, processed_file)

    # Load the file into a DataFrame
    df = pd.read_excel(processed_file_path)

    # Extract the year from the file name (e.g., "2023_processed.xlsx" -> 2023)
    year = processed_file.split('_')[0]

    # Add a new column 'conf_year' based on the year
    df['conf_year'] = year

    # Append the DataFrame to the list
    dfs.append(df)

# Concatenate all DataFrames along rows (axis=0)
concatenated_df = pd.concat(dfs, ignore_index=True)

# Save the concatenated DataFrame to a new Excel file
output_file_path = '/content/drive/My Drive/clean/ashoka/processed/concatenated_data.xlsx'
concatenated_df.to_excel(output_file_path, index=False)

# Print the result and download the file
files.download(output_file_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Cleaning DSE Winter Schools .

In [ ]:
# Load the Excel file with multiple sheets
file_path = '/content/drive/My Drive/DSE/DSE_clean/clean/DSE2023.xlsx'
xls = pd.ExcelFile(file_path)

In [ ]:
from google.colab import files

output_file = "dse2023_updated.xlsx"


# Load all sheets into a dictionary of DataFrames
xls = pd.ExcelFile(file_path)
sheets = {sheet_name: xls.parse(sheet_name) for sheet_name in xls.sheet_names}

# Select the 'Authorwise 2018' sheet for processing
authorwise_df = sheets['presentations']

# Add new columns with specified values
authorwise_df['conf'] = 'DSE Winter School'
authorwise_df['conf_year'] = '2023'

# Rename columns
authorwise_df.rename(columns={
            'Presentation title': 'paper_title',
            'Session topic': 'theme_isi',
            'Author numbers': 'authors_total'
}, inplace=True)

# Create new columns for male and female authors
authorwise_df['authors_total_male'] = 0
authorwise_df['authors_total_female'] = 0

# Count male and female authors per paper_title
def count_authors_by_gender(group):
    male_count = (group['Gender'] == 'Male').sum()
    female_count = (group['Gender'] == 'Female').sum()
    group['authors_total_male'] = male_count
    group['authors_total_female'] = female_count
    return group

authorwise_df = authorwise_df.groupby('paper_title').apply(count_authors_by_gender)

# Save the updated DataFrame back to the original dictionary of sheets
sheets['presentations'] = authorwise_df

# Write all sheets back to an Excel file
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# Save the file to Google Drive
files.download(output_file)

print(f"Updated file saved as {output_file} and downloaded to your system.")

<ipython-input-70-2d05bbdde631>:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  authorwise_df = authorwise_df.groupby('paper_title').apply(count_authors_by_gender)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Updated file saved as dse2023_updated.xlsx and downloaded to your system.


In [ ]:
import pandas as pd
import re

# Load the spreadsheet
file_path = '/content/drive/My Drive/DSE/DSE_clean/clean/dse2023_updated.xlsx'
sheet_name = 'presentations'

# Load the Authorwise 2018 sheet
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Remove characters and text in between '():' in theme_isi column
#df['theme_isi'] = df['theme_isi'].apply(lambda x: re.sub(r'\([a-zA-Z]\):', '', x))

# Create new columns with NaN values
new_columns = [
    'presenter_autho', 'author_name_1', 'author_gender_1', 'author_qual_1', 'author_qual_1_o',
    'department_1', 'author_insti_1', 'author_insti_country1', 'phd_comp_1', 'phd_subject_1',
    'phd_insti_1', 'phd_inscountry_1', 'phd_yr_1', 'masters_comp_1', 'masters_insti_1',
    'masters_country1', 'masters_yr_1', 'underg_subject_1', 'underg_insti_1', 'underg_country_1',
    'underg_yr_1', 'email1_1', 'email2_1', 'author_name_2', 'author_gender_2', 'author_qual_2',
    'author_qual_2_o', 'department_2', 'author_insti_2', 'author_insti_country2', 'phd_comp_2',
    'phd_subject_2', 'phd_insti_2', 'phd_inscountry_2', 'phd_yr_2', 'masters_comp_2',
    'masters_insti_2', 'masters_country2', 'masters_yr_2', 'underg_subject_2', 'underg_insti_2',
    'underg_country_2', 'underg_yr_2', 'email1_2', 'email2_2', 'author_name_3', 'author_gender_3',
    'author_qual_3', 'author_qual_3_o', 'department_3', 'author_insti_3', 'author_insti_country3',
    'phd_comp_3', 'phd_subject_3', 'phd_insti_3', 'phd_inscountry_3', 'phd_yr_3', 'masters_comp_3',
    'masters_insti_3', 'masters_country3', 'masters_yr_3', 'underg_subject_3', 'underg_insti_3',
    'underg_country_3', 'underg_yr_3', 'email1_3', 'email2_3', 'author_name_4', 'author_gender_4',
    'author_qual_4', 'author_qual_4_o', 'department_4', 'author_insti_4', 'author_insti_country4',
    'phd_comp_4', 'phd_subject_4', 'phd_insti_4', 'phd_inscountry_4', 'phd_yr_4', 'masters_comp_4',
    'masters_insti_4', 'masters_country4', 'masters_yr_4', 'underg_subject_4', 'underg_insti_4',
    'underg_country_4', 'underg_yr_4', 'email1_4', 'email2_4', 'author_name_5', 'author_gender_5',
    'author_qual_5', 'author_qual_5_o', 'department_5', 'author_insti_5', 'author_insti_country5',
    'phd_comp_5', 'phd_subject_5', 'phd_insti_5', 'phd_inscountry_5', 'phd_yr_5', 'masters_comp_5',
    'masters_insti_5', 'masters_country5', 'masters_yr_5', 'underg_subject_5', 'underg_insti_5',
    'underg_country_5', 'underg_yr_5', 'email1_5', 'email2_5', 'author_name_6', 'author_gender_6',
    'author_qual_6', 'author_qual_6_o', 'department_6', 'author_insti_6', 'author_insti_country6',
    'phd_comp_6', 'phd_subject_6', 'phd_insti_6', 'phd_inscountry_6', 'phd_yr_6', 'masters_comp_6',
    'masters_insti_6', 'masters_country6', 'masters_yr_6', 'underg_subject_6', 'underg_insti_6',
    'underg_country_6', 'underg_yr_6', 'email1_6', 'email2_6', 'author_name_7', 'author_gender_7',
    'author_qual_7', 'author_qual_7_o', 'department_7', 'author_insti_7', 'author_insti_country7',
    'phd_comp_7', 'phd_subject_7', 'phd_insti_7', 'phd_inscountry_7', 'phd_yr_7', 'masters_comp_7',
    'masters_insti_7', 'masters_country7', 'masters_yr_7', 'underg_subject_7', 'underg_insti_7',
    'underg_country_7', 'underg_yr_7', 'email1_7', 'Email2_7'
]

# Use pd.concat to add all new columns at once to avoid fragmentation
df = pd.concat([df, pd.DataFrame(columns=new_columns)], axis=1)

# Prepare a new DataFrame to hold the transformed data
transformed_data = []

# Fill the new DataFrame with data
unique_papers = df['paper_title'].unique()
for paper in unique_papers:
    paper_data = df[df['paper_title'] == paper]
    authors = paper_data['Authors'].unique()
    new_row = paper_data.iloc[0].to_dict()

    for j, author in enumerate(authors):
        author_idx = j + 1
        # Check if the filtered DataFrame is empty before accessing .iloc[0]
        filtered_author_data = paper_data[paper_data['Authors'] == author]
        if not filtered_author_data.empty:
            author_info = filtered_author_data.iloc[0]

        new_row[f'author_name_{author_idx}'] = author
        new_row[f'author_insti_{author_idx}'] = author_info['Author Affiliation']
        new_row[f'author_insti_country{author_idx}'] = author_info['Affiliation Location']
        new_row[f'author_gender_{author_idx}'] = author_info['Gender']
        #new_row[f'author_qual_{author_idx}'] = author_info['Author Qualification']
        #new_row[f'department_{author_idx}'] = author_info['Department']
        #new_row[f'phd_comp_{author_idx}'] = author_info['PhD Completed']
        #new_row[f'phd_subject_{author_idx}'] = author_info['PhD Subject']
        #new_row[f'phd_insti_{author_idx}'] = author_info['PhD Institute']
        #new_row[f'phd_inscountry_{author_idx}'] = author_info['PhD Institute Country']
        #new_row[f'phd_yr_{author_idx}'] = author_info['PhD Year']
        #new_row[f'masters_comp_{author_idx}'] = author_info['Masters Completed']
        #new_row[f'masters_insti_{author_idx}'] = author_info['Masters Institute']
        #new_row[f'masters_country_{author_idx}'] = author_info['Masters Country']
        #new_row[f'masters_yr_{author_idx}'] = author_info['Masters Year']
        #new_row[f'underg_subject_{author_idx}'] = author_info['Undergraduate Subject']
        #new_row[f'underg_insti_{author_idx}'] = author_info['Undergraduate Institute']
        #new_row[f'underg_country_{author_idx}'] = author_info['Undergraduate Country']
        #new_row[f'underg_yr_{author_idx}'] = author_info['Undergraduate Year']
        #new_row[f'email1_{author_idx}'] = author_info['Email 1']
        #new_row[f'email2_{author_idx}'] = author_info['Email 2']

    transformed_data.append(new_row)

transformed_df = pd.DataFrame(transformed_data)

# Save the updated dataframe back to Excel
updated_file_path = '/content/drive/My Drive/DSE/DSE_clean/clean/dse2023_updated1.xlsx'
with pd.ExcelWriter(updated_file_path, engine='openpyxl', mode='w') as writer:
    transformed_df.to_excel(writer, sheet_name=sheet_name, index=False)

# Confirm the file has been updated
print(f"Updated file saved to {updated_file_path}")


Updated file saved to /content/drive/My Drive/DSE/DSE_clean/clean/dse2023_updated1.xlsx


In [ ]:
import pandas as pd

# Load ISI2018_updated file
isi2018_file_path = '/content/drive/My Drive/DSE/DSE_clean/clean/dse2023_updated.xlsx'
isi2018_sheet_name = 'presentations'
df_isi2018 = pd.read_excel(isi2018_file_path, sheet_name=isi2018_sheet_name)

# Load ISI2019_updated file
isi2019_file_path = '/content/drive/My Drive/DSE/DSE_clean/clean/dse2023_updated1.xlsx'
isi2019_sheet_name = 'presentations'
df_isi2019 = pd.read_excel(isi2019_file_path, sheet_name=isi2019_sheet_name)

# Check the number of unique paper_title in ISI2018
unique_papers_2018 = df_isi2018['paper_title'].nunique()
print(f"Number of unique paper_title in ISI2023: {unique_papers_2018}")

# Check the number of unique paper_title in ISI2019
unique_papers_2018up = df_isi2019['paper_title'].nunique()
print(f"Number of unique paper_title in ISI2023updated: {unique_papers_2018up}")


Number of unique paper_title in ISI2023: 181
Number of unique paper_title in ISI2023updated: 181


MErging

In [ ]:
import pandas as pd

# File paths and expected sheet names
file_paths = {
    'dse2020_updated1.xlsx': '/content/drive/My Drive/DSE/DSE_clean/clean/dse2020_updated1.xlsx',
    'dse2021_updated1.xlsx': '/content/drive/My Drive/DSE/DSE_clean/clean/dse2021_updated1.xlsx',
    #'ISI2020_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2020_updated1.xlsx',
    'dse2022_updated1.xlsx': '/content/drive/My Drive/DSE/DSE_clean/clean/dse2022_updated1.xlsx',
    'dse2023_updated1.xlsx': '/content/drive/My Drive/DSE/DSE_clean/clean/dse2023_updated1.xlsx',
    #'ISI2023_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2023_updated1.xlsx'
}

final_file_path = '/content/drive/My Drive/DSE/DSE_clean/clean/dse2020-2023.xlsx'

# Load all files and check columns
data_frames = []
reference_columns = None
mismatch_files = []

for file_name, file_path in file_paths.items():
    df = pd.read_excel(file_path)
    data_frames.append(df)
    if reference_columns is None:
        reference_columns = set(df.columns)
    else:
        current_columns = set(df.columns)
        if current_columns != reference_columns:
            mismatch_files.append((file_name, list(reference_columns.symmetric_difference(current_columns))))

if mismatch_files:
    for file, mismatched_columns in mismatch_files:
        print(f"Columns not matching in file {file}: {mismatched_columns}")
else:
    print("All files have matching columns.")

# Concatenate all data frames
combined_df = pd.concat(data_frames, ignore_index=True)

# Sort by 'conf_year'
combined_df.sort_values(by='conf_year', inplace=True)

# Save the concatenated data to a new Excel file with separate sheets
with pd.ExcelWriter(final_file_path, engine='openpyxl') as writer:
    for file_name, file_path in file_paths.items():
        df = pd.read_excel(file_path)
        sheet_name = file_name.replace('.xlsx', '').replace('updated1', '').strip('_')
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    combined_df.to_excel(writer, sheet_name='Paper Presenters', index=False)

# Confirm the file has been updated
print(f"Concatenated file saved to {final_file_path}")


Columns not matching in file dse2021_updated1.xlsx: ['Unnamed: 6']
Concatenated file saved to /content/drive/My Drive/DSE/DSE_clean/clean/dse2020-2023.xlsx


# Cleaning SAU

In [ ]:
# Install necessary libraries
!pip install pandas openpyxl

# Import required libraries
import pandas as pd
from google.colab import drive

# Path to your Excel file in Google Drive
file_path = '/content/drive/My Drive/SAU/sau_all.xlsx'  # Change this to the actual path of your file

# Load the Excel file with all sheets
excel_data = pd.ExcelFile(file_path)

# Get all sheet names
sheet_names = excel_data.sheet_names

# Loop through each sheet, save it as a separate Excel file, and download
for sheet in sheet_names:
    # Read each sheet
    df = excel_data.parse(sheet)

    # Define the output file name (use the sheet name)
    output_file = f"{sheet}.xlsx"

    # Save the sheet to a new Excel file
    df.to_excel(output_file, index=False)

    # Download the file
    from google.colab import files
    files.download(output_file)

print("All sheets have been processed and are ready for download!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All sheets have been processed and are ready for download!


In [ ]:
# Load the Excel file with multiple sheets
file_path = '/content/drive/My Drive/SAU/2019.xlsx'
xls = pd.ExcelFile(file_path)

In [ ]:
from google.colab import files
import pandas as pd

output_file = "sau2019_updated.xlsx"

# Load all sheets into a dictionary of DataFrames
xls = pd.ExcelFile(file_path)
sheets = {sheet_name: xls.parse(sheet_name) for sheet_name in xls.sheet_names}

# Select the first sheet for processing (since sheet names are not used)
first_sheet_name = list(sheets.keys())[0]
authorwise_df = sheets[first_sheet_name]

# Rename the column 'Author\'s Gender(Female=1,Male=0)' into 'Gender'
#df.rename(columns={}, inplace=True)


# Add new columns with specified values
authorwise_df['conf'] = 'SAU'
authorwise_df['conf_year'] = '2019'

# Rename columns
authorwise_df.rename(columns={
    'Paper Title': 'paper_title',
    'Topic': 'theme_isi',
    'Number of Authors': 'authors_total'
    #"Author\'s Gender(Female=1,Male=0): 'Gender'
}, inplace=True)

# Create new columns for male and female authors
authorwise_df['authors_total_male'] = 0
authorwise_df['authors_total_female'] = 0

# Count male and female authors per paper_title
def count_authors_by_gender(group):
    male_count = (group['Gender'] == 0).sum()
    female_count = (group['Gender'] == 1).sum()
    group['authors_total_male'] = male_count
    group['authors_total_female'] = female_count
    return group

authorwise_df = authorwise_df.groupby('paper_title').apply(count_authors_by_gender)

# Save the updated DataFrame back to the original dictionary of sheets
sheets[first_sheet_name] = authorwise_df

# Write all sheets back to an Excel file
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# Save the file to Google Drive
files.download(output_file)

print(f"Updated file saved as {output_file} and downloaded to your system.")


<ipython-input-42-6c34f9500a94>:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  authorwise_df = authorwise_df.groupby('paper_title').apply(count_authors_by_gender)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Updated file saved as sau2019_updated.xlsx and downloaded to your system.


In [ ]:
import pandas as pd
import re

# Load the spreadsheet
file_path = '/content/drive/My Drive/SAU/sau2019_updated.xlsx'
# Load all sheets into a dictionary of DataFrames
xls = pd.ExcelFile(file_path)
sheets = {sheet_name: xls.parse(sheet_name) for sheet_name in xls.sheet_names}

# Select the first sheet for processing (since sheet names are not used)
first_sheet_name = list(sheets.keys())[0]
authorwise_df = sheets[first_sheet_name]

# Remove characters and text in between '():' in theme_isi column
#df['theme_isi'] = df['theme_isi'].apply(lambda x: re.sub(r'\([a-zA-Z]\):', '', x))

# Create new columns with NaN values
new_columns = [
    'presenter_autho', 'author_name_1', 'author_gender_1', 'author_qual_1', 'author_qual_1_o',
    'department_1', 'author_insti_1', 'author_insti_country1', 'phd_comp_1', 'phd_subject_1',
    'phd_insti_1', 'phd_inscountry_1', 'phd_yr_1', 'masters_comp_1', 'masters_insti_1',
    'masters_country1', 'masters_yr_1', 'underg_subject_1', 'underg_insti_1', 'underg_country_1',
    'underg_yr_1', 'email1_1', 'email2_1', 'author_name_2', 'author_gender_2', 'author_qual_2',
    'author_qual_2_o', 'department_2', 'author_insti_2', 'author_insti_country2', 'phd_comp_2',
    'phd_subject_2', 'phd_insti_2', 'phd_inscountry_2', 'phd_yr_2', 'masters_comp_2',
    'masters_insti_2', 'masters_country2', 'masters_yr_2', 'underg_subject_2', 'underg_insti_2',
    'underg_country_2', 'underg_yr_2', 'email1_2', 'email2_2', 'author_name_3', 'author_gender_3',
    'author_qual_3', 'author_qual_3_o', 'department_3', 'author_insti_3', 'author_insti_country3',
    'phd_comp_3', 'phd_subject_3', 'phd_insti_3', 'phd_inscountry_3', 'phd_yr_3', 'masters_comp_3',
    'masters_insti_3', 'masters_country3', 'masters_yr_3', 'underg_subject_3', 'underg_insti_3',
    'underg_country_3', 'underg_yr_3', 'email1_3', 'email2_3', 'author_name_4', 'author_gender_4',
    'author_qual_4', 'author_qual_4_o', 'department_4', 'author_insti_4', 'author_insti_country4',
    'phd_comp_4', 'phd_subject_4', 'phd_insti_4', 'phd_inscountry_4', 'phd_yr_4', 'masters_comp_4',
    'masters_insti_4', 'masters_country4', 'masters_yr_4', 'underg_subject_4', 'underg_insti_4',
    'underg_country_4', 'underg_yr_4', 'email1_4', 'email2_4', 'author_name_5', 'author_gender_5',
    'author_qual_5', 'author_qual_5_o', 'department_5', 'author_insti_5', 'author_insti_country5',
    'phd_comp_5', 'phd_subject_5', 'phd_insti_5', 'phd_inscountry_5', 'phd_yr_5', 'masters_comp_5',
    'masters_insti_5', 'masters_country5', 'masters_yr_5', 'underg_subject_5', 'underg_insti_5',
    'underg_country_5', 'underg_yr_5', 'email1_5', 'email2_5', 'author_name_6', 'author_gender_6',
    'author_qual_6', 'author_qual_6_o', 'department_6', 'author_insti_6', 'author_insti_country6',
    'phd_comp_6', 'phd_subject_6', 'phd_insti_6', 'phd_inscountry_6', 'phd_yr_6', 'masters_comp_6',
    'masters_insti_6', 'masters_country6', 'masters_yr_6', 'underg_subject_6', 'underg_insti_6',
    'underg_country_6', 'underg_yr_6', 'email1_6', 'email2_6', 'author_name_7', 'author_gender_7',
    'author_qual_7', 'author_qual_7_o', 'department_7', 'author_insti_7', 'author_insti_country7',
    'phd_comp_7', 'phd_subject_7', 'phd_insti_7', 'phd_inscountry_7', 'phd_yr_7', 'masters_comp_7',
    'masters_insti_7', 'masters_country7', 'masters_yr_7', 'underg_subject_7', 'underg_insti_7',
    'underg_country_7', 'underg_yr_7', 'email1_7', 'Email2_7'
]

# Use pd.concat to add all new columns at once to avoid fragmentation
df = pd.concat([df, pd.DataFrame(columns=new_columns)], axis=1)

# Prepare a new DataFrame to hold the transformed data
transformed_data = []

# Fill the new DataFrame with data
unique_papers = df['paper_title'].unique()
for paper in unique_papers:
    paper_data = df[df['paper_title'] == paper]
    authors = paper_data['Author'].unique()
    new_row = paper_data.iloc[0].to_dict()

    for j, author in enumerate(authors):
        author_idx = j + 1
        # Check if the filtered DataFrame is empty before accessing .iloc[0]
        filtered_author_data = paper_data[paper_data['Author'] == author]
        if not filtered_author_data.empty:
            author_info = filtered_author_data.iloc[0]

        new_row[f'author_name_{author_idx}'] = author
        new_row[f'author_insti_{author_idx}'] = author_info['Institute/Affiliation']
        new_row[f'author_insti_country{author_idx}'] = author_info['Author/Presenter\'s Country']
        new_row[f'author_gender_{author_idx}'] = author_info['Gender']
        #new_row[f'author_qual_{author_idx}'] = author_info['Author Qualification']
        #new_row[f'department_{author_idx}'] = author_info['Department']
        #new_row[f'phd_comp_{author_idx}'] = author_info['PhD Completed']
        #new_row[f'phd_subject_{author_idx}'] = author_info['PhD Subject']
        #new_row[f'phd_insti_{author_idx}'] = author_info['PhD Institute']
        #new_row[f'phd_inscountry_{author_idx}'] = author_info['PhD Institute Country']
        #new_row[f'phd_yr_{author_idx}'] = author_info['PhD Year']
        #new_row[f'masters_comp_{author_idx}'] = author_info['Masters Completed']
        #new_row[f'masters_insti_{author_idx}'] = author_info['Masters Institute']
        #new_row[f'masters_country_{author_idx}'] = author_info['Masters Country']
        #new_row[f'masters_yr_{author_idx}'] = author_info['Masters Year']
        #new_row[f'underg_subject_{author_idx}'] = author_info['Undergraduate Subject']
        #new_row[f'underg_insti_{author_idx}'] = author_info['Undergraduate Institute']
        #new_row[f'underg_country_{author_idx}'] = author_info['Undergraduate Country']
        #new_row[f'underg_yr_{author_idx}'] = author_info['Undergraduate Year']
        #new_row[f'email1_{author_idx}'] = author_info['Email 1']
        #new_row[f'email2_{author_idx}'] = author_info['Email 2']

    transformed_data.append(new_row)

transformed_df = pd.DataFrame(transformed_data)

# Save the updated dataframe back to Excel
updated_file_path = '/content/drive/My Drive/SAU/sau2019_updated1.xlsx'
with pd.ExcelWriter(updated_file_path, engine='openpyxl', mode='w') as writer:
    transformed_df.to_excel(writer, sheet_name=sheet_name, index=False)

# Confirm the file has been updated
print(f"Updated file saved to {updated_file_path}")


Updated file saved to /content/drive/My Drive/SAU/sau2019_updated1.xlsx


In [ ]:
import pandas as pd

# Load ISI2018_updated file
isi2018_file_path = '/content/drive/My Drive/SAU/sau2015_updated.xlsx'
isi2018_sheet_name = 'presentations'
df_isi2018 = pd.read_excel(isi2018_file_path, sheet_name=isi2018_sheet_name)

# Load ISI2019_updated file
isi2019_file_path = '/content/drive/My Drive/SAU/sau2015_updated1.xlsx'
isi2019_sheet_name = 'presentations'
df_isi2019 = pd.read_excel(isi2019_file_path, sheet_name=isi2019_sheet_name)

# Check the number of unique paper_title in ISI2018
unique_papers_2018 = df_isi2018['paper_title'].nunique()
print(f"Number of unique paper_title in ISI2023: {unique_papers_2018}")

# Check the number of unique paper_title in ISI2019
unique_papers_2018up = df_isi2019['paper_title'].nunique()
print(f"Number of unique paper_title in ISI2023updated: {unique_papers_2018up}")


ValueError: Worksheet named 'presentations' not found

MErging

In [ ]:
import pandas as pd

# File paths and expected sheet names
file_paths = {
    'sau2015_updated1.xlsx': '/content/drive/My Drive/SAU/sau2015_updated1.xlsx',
    'sau2016_updated1.xlsx': '/content/drive/My Drive/SAU/sau2016_updated1.xlsx',
    #'ISI2020_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2020_updated1.xlsx',
    'sau2018_updated1.xlsx': '/content/drive/My Drive/SAU/sau2018_updated1.xlsx',
    'sau2019_updated1.xlsx': '/content/drive/My Drive//SAU/sau2019_updated1.xlsx',
    #'ISI2023_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2023_updated1.xlsx'
}

final_file_path = '/content/drive/My Drive/DSE/DSE_clean/clean/sau2015-2019.xlsx'

# Load all files and check columns
data_frames = []
reference_columns = None
mismatch_files = []

for file_name, file_path in file_paths.items():
    df = pd.read_excel(file_path)
    data_frames.append(df)
    if reference_columns is None:
        reference_columns = set(df.columns)
    else:
        current_columns = set(df.columns)
        if current_columns != reference_columns:
            mismatch_files.append((file_name, list(reference_columns.symmetric_difference(current_columns))))

if mismatch_files:
    for file, mismatched_columns in mismatch_files:
        print(f"Columns not matching in file {file}: {mismatched_columns}")
else:
    print("All files have matching columns.")

# Concatenate all data frames
combined_df = pd.concat(data_frames, ignore_index=True)

# Sort by 'conf_year'
combined_df.sort_values(by='conf_year', inplace=True)

# Save the concatenated data to a new Excel file with separate sheets
with pd.ExcelWriter(final_file_path, engine='openpyxl') as writer:
    for file_name, file_path in file_paths.items():
        df = pd.read_excel(file_path)
        sheet_name = file_name.replace('.xlsx', '').replace('updated1', '').strip('_')
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    combined_df.to_excel(writer, sheet_name='Paper Presenters', index=False)

# Confirm the file has been updated
print(f"Concatenated file saved to {final_file_path}")


Columns not matching in file sau2016_updated1.xlsx: ["Author/Presenter's Country", 'Affiliation Location']
Columns not matching in file sau2018_updated1.xlsx: ["Author/Presenter's Country", 'Affiliation Location']
Columns not matching in file sau2019_updated1.xlsx: ["Author/Presenter's Country", 'Affiliation Location']
Concatenated file saved to /content/drive/My Drive/DSE/DSE_clean/clean/sau2015-2019.xlsx


# Merging all

In [ ]:
import pandas as pd

# File paths and expected sheet names
file_paths = {
    'sau2015-2019.xlsx': '/content/drive/My Drive/conferences/sau2015-2019.xlsx',
    'dse2020-2023.xlsx': '/content/drive/My Drive/conferences/dse2020-2023.xlsx',
    #'ISI2020_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2020_updated1.xlsx',
    'ISIconferences2018-2023.xlsx': '/content/drive/My Drive/conferences/ISIconferences2018-2023.xlsx',
    'auconf_updated1.xlsx': '/content/drive/My Drive/conferences/auconf_updated1.xlsx',
    #'ISI2023_updated1.xlsx': '/content/drive/My Drive/ISI/ISIclean/ISI2023_updated1.xlsx'
}

final_file_path = '/content/drive/My Drive/conferences/conferences_clean.xlsx'

# Load all files and check columns
data_frames = []
reference_columns = None
mismatch_files = []

for file_name, file_path in file_paths.items():
    df = pd.read_excel(file_path)
    data_frames.append(df)
    if reference_columns is None:
        reference_columns = set(df.columns)
    else:
        current_columns = set(df.columns)
        if current_columns != reference_columns:
            mismatch_files.append((file_name, list(reference_columns.symmetric_difference(current_columns))))

if mismatch_files:
    for file, mismatched_columns in mismatch_files:
        print(f"Columns not matching in file {file}: {mismatched_columns}")
else:
    print("All files have matching columns.")

# Concatenate all data frames
combined_df = pd.concat(data_frames, ignore_index=True)

# Sort by 'conf_year'
combined_df.sort_values(by='conf_year', inplace=True)

# Save the concatenated data to a new Excel file with separate sheets
with pd.ExcelWriter(final_file_path, engine='openpyxl') as writer:
    for file_name, file_path in file_paths.items():
        df = pd.read_excel(file_path)
        sheet_name = file_name.replace('.xlsx', '').replace('updated1', '').strip('_')
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    combined_df.to_excel(writer, sheet_name='Paper Presenters', index=False)

# Confirm the file has been updated
print(f"Concatenated file saved to {final_file_path}")


Columns not matching in file dse2020-2023.xlsx: ['Affiliation Location India', 'Authors', 'Source used for determining gender', 'Author', 'Author Affiliation', 'Institute/Affiliation']
Columns not matching in file ISIconferences2018-2023.xlsx: ['author_insti_country97', 'author_gender_70', 'author_insti_67', 'author_gender_42', 'author_insti_country50', 'author_insti_country61', 'author_insti_country77', 'author_insti_26', 'author_insti_78', 'Affiliation Country', 'author_gender_25', 'author_insti_84', 'author_gender_34', 'author_insti_18', 'author_name_9', 'author_insti_12', 'author_name_92', 'author_gender_86', 'author_gender_72', 'author_insti_44', 'author_insti_63', 'author_insti_country39', 'author_insti_54', 'author_insti_9', 'author_gender_14', 'author_insti_country10', 'author_insti_country85', 'author_insti_86', 'author_insti_60', 'author_insti_country15', 'author_insti_43', 'author_insti_19', 'author_name_43', 'author_name_64', 'author_name_68', 'author_insti_45', 'author_nam

## Save dataset

In [ ]:
# Save the cleaned and updated DataFrame to a new CSV
df1_extended.to_csv("/content/drive/My Drive/clean/cleaner/career_extended_unsorted.csv", index=False)

# Cleaning master_dataset

In [ ]:
# Import necessary libraries
import pandas as pd

# Define file paths
file1_path = '/content/drive/My Drive/clean/clean/master_dataset.csv'
file2_path = '/content/drive/My Drive/clean/clean/temporarycareer.csv'

# Load CSV files
master_dataset = pd.read_csv(file1_path)
temporarycareer = pd.read_csv(file2_path)

# List columns for each file
print("Columns in master_dataset.csv:")
print(master_dataset.columns.tolist())

print("\nColumns in temporarycareer.csv:")
print(temporarycareer.columns.tolist())


Columns in master_dataset.csv:
['Name', 'revised_gender', 'UniversityType', 'Institution', 'archive_link', 'college_url', 'scholar_profile_url', 'State', 'Specialisation', 'AreaofInterest', 'UGDegree', 'UGDepartment', 'UGInstitute', 'PGDegree', 'PGDepartment', 'PGInstitute', 'MPhil', 'MPhilDepartment', 'MPhilInstitute', 'Doctoral', 'DoctoralDepartment', 'DoctoralInstitute', 'FirstJobPosition', 'FirstJobInstitution', 'FirstJobDuration', 'SecondJobPosition', 'SecondJobInstitution', 'SecondJobDuration', 'ThirdJobPosition', 'ThirdJobInstitution', 'ThirdJobDuration', 'FourthJobPosition', 'FourthJobInstitution', 'FourthJobDuration', 'FifthJobPosition', 'FifthJobInstitution', 'FifthJobDuration', 'unique_id', 'university_type', 'name', 'institution', 'UniversityType_id', 'State_id', 'Name_id', 'Institution_id', 'Current_Designation', 'dept', 'state', 'CurrentDesignation_r', 'archive_timeline', 'identifier', 'names_scholar', 'position', 'field', 'total_citation', 'citations_dynmc', 'citation_5y

Drop unnecessary variables from temporary career

In [ ]:
# Import necessary library
import os

# List of columns to drop
columns_to_drop = [
    'name', 'names_scholar', 'Institution', 'scholar_profile_url',
    'linkedin_site_manual', 'linkedin', 'linkedinProfileUrl', 'linkedinProfile',
    'location', 'imgUrl', 'firstName', 'lastName', 'connectionDegree', 'vmid',
    'userId', 'linkedinSalesNavigatorUrl', 'connectionsUrl', 'mutualConnectionsUrl',
    'mutualConnectionsText', 'isOpenLink', 'isOpenToWork', 'isHiring',
    'companyDomainName', 'phoneNumberFromDropContact', 'websiteFromDropContact',
    'website', 'companyWebsite', 'allSkills', 'skill1', 'endorsement1', 'skill2',
    'endorsement2', 'skill3', 'endorsement3', 'skill4', 'endorsement4', 'skill5',
    'endorsement5', 'skill6', 'endorsement6', 'baseUrl', 'profileId', 'timestamp',
    'recordId', 'companyLinkedInUrl', 'error', 'jobDescription2', 'schoolDescription2',
    'civilityFromDropContact', 'birthday', 'subscribers', 'jobDescription',
    'mailFromDropcontact', 'qualificationFromDropContact', 'mail',
    'nbEmployeesFromDropContact', 'connectedOn', 'facebookUrl'
]

# Drop the specified columns
temporarycareer = temporarycareer.drop(columns=columns_to_drop, errors='ignore')

# Save the updated DataFrame back to the same location in Google Drive
file_path = '/content/drive/My Drive/clean/clean/temporarycareer1.csv'
temporarycareer.to_csv(file_path, index=False)

print(f"Updated CSV saved successfully at {file_path}.")


Updated CSV saved successfully at /content/drive/My Drive/clean/clean/temporarycareer1.csv.


In [ ]:
print("\nColumns in temporarycareer.csv:")
print(temporarycareer.columns.tolist())


Columns in temporarycareer.csv:
['Name', 'email', 'description', 'headline', 'fullName', 'connectionsCount', 'company', 'companyUrl', 'jobTitle', 'jobLocation', 'jobDateRange', 'jobStartedSince', 'jobIsCurrent', 'jobDuration', 'company2', 'companyUrl2', 'jobTitle2', 'jobLocation2', 'jobDateRange2', 'jobStartedSince2', 'jobIsCurrent2', 'jobDuration2', 'school', 'schoolUrl', 'schoolDegree', 'schoolDescription', 'schoolDateRange', 'school2', 'schoolUrl2', 'schoolDegree2', 'schoolDateRange2', 'professionalEmail']


Prepare to merge

In [ ]:
# Check for matches between 'fullName' in temporarycareer and 'Name' in master_dataset
matches = temporarycareer['Name'].isin(master_dataset['Name'])

# Count the number of matches
num_matches = matches.sum()

print(f"Number of matches between 'fullName' in temporarycareer and 'Name' in master_dataset: {num_matches}")


Number of matches between 'fullName' in temporarycareer and 'Name' in master_dataset: 886


In [ ]:
# Count the total number of rows in each dataset
rows_in_temporarycareer1 = temporarycareer.shape[0]
rows_in_master_dataset = master_dataset.shape[0]

# Count the number of matches between 'fullName' in temporarycareer and 'Name' in master_dataset
matches = temporarycareer['Name'].isin(master_dataset['Name'])
num_matches = matches.sum()

# Print the results
print(f"Number of rows in temporarycareer: {rows_in_temporarycareer}")
print(f"Number of rows in master_dataset: {rows_in_master_dataset}")
print(f"Number of matches between 'fullName' in temporarycareer and 'Name' in master_dataset: {num_matches}")


Number of rows in temporarycareer: 886
Number of rows in master_dataset: 2636
Number of matches between 'fullName' in temporarycareer and 'Name' in master_dataset: 886


In [ ]:
import pandas as pd

# Filter rows where 'linkedin_site_manual' is not empty or NaN
filtered_master_dataset = master_dataset[master_dataset['linkedin_site_manual'].notna() & (master_dataset['linkedin_site_manual'] != '')]

# Save the filtered DataFrame as a new CSV in the same location
filtered_master_dataset.to_csv('/content/drive/My Drive/clean/clean/master_dataset_filtered.csv', index=False)

print("Filtered CSV saved successfully!")


Filtered CSV saved successfully!


In [ ]:
# Count the number of rows
num_rows = filtered_master_dataset.shape[0]

print(f"Number of rows in the filtered master dataset: {num_rows}")


Number of rows in the filtered master dataset: 903


Rename columns in temporarycareer1


In [ ]:
import pandas as pd

# Load the CSV file
temporarycareer1 = pd.read_csv('/content/drive/My Drive/clean/clean/temporarycareer1.csv')

# Rename the columns
rename_columns = {
    'company': 'FirstJobInstitution',
    'companyUrl': 'FirstJobInstitution url',
    'jobTitle': 'FirstJobPosition',
    'jobDuration': 'FirstJobDuration',
    'company2': 'SecondJobInstitution',
    'companyUrl2': 'SecondJobInstitution url',
    'jobTitle2': 'SecondJobPosition',
    'jobDuration2': 'SecondJobDuration'
}

temporarycareer1.rename(columns=rename_columns, inplace=True)

# Save the updated CSV file with the same name
temporarycareer1.to_csv('/content/drive/My Drive/clean/clean/temporarycareer1.csv', index=False)

print("Columns renamed and CSV saved successfully!")


Columns renamed and CSV saved successfully!


Merging

In [ ]:
import pandas as pd

# Load both datasets
master_dataset = pd.read_csv('/content/drive/My Drive/clean/clean/master_dataset.csv')
temporarycareer1 = pd.read_csv('/content/drive/My Drive/clean/clean/temporarycareer1.csv')

# Step 1: Strip extra spaces from column names (if any)
master_dataset.columns = master_dataset.columns.str.strip()
temporarycareer1.columns = temporarycareer1.columns.str.strip()

# Step 2: Identify columns in temporarycareer1 that are not in master_dataset
columns_in_master = master_dataset.columns
columns_in_temp = temporarycareer1.columns

# Find columns that are in temporarycareer1 but not in master_dataset
new_columns = [col for col in columns_in_temp if col not in columns_in_master]

# Step 3: Add the new columns from temporarycareer1 to master_dataset
for col in new_columns:
    master_dataset[col] = temporarycareer1[col]

# Step 4: Merge the datasets based on the 'Name' column
merged_df = pd.merge(master_dataset, temporarycareer1, on='Name', how='left')

# Step 5: Save the updated master dataset with the new columns
merged_df.to_csv('/content/drive/My Drive/clean/clean/master_dataset_updated.csv', index=False)

# Output number of rows in the updated master dataset
print(f"Rows in updated master dataset: {len(merged_df)}")


Rows in updated master dataset: 2660


In [ ]:
# Count the number of rows
num_rows = final_df.shape[0]

print(f"Number of rows in the filtered master dataset: {num_rows}")


In [ ]:
import pandas as pd

# Load both datasets
master_dataset = pd.read_csv('/content/drive/My Drive/clean/clean/master_dataset.csv')
temporarycareer1 = pd.read_csv('/content/drive/My Drive/clean/clean/temporarycareer1.csv')

# Step 1: Strip extra spaces from column names (if any)
master_dataset.columns = master_dataset.columns.str.strip()
temporarycareer1.columns = temporarycareer1.columns.str.strip()

# Step 2: Print columns of both datasets
print("Columns in master_dataset:")
print(master_dataset.columns)

print("\nColumns in temporarycareer1:")
print(temporarycareer1.columns)


Columns in master_dataset:
Index(['Name', 'revised_gender', 'UniversityType', 'Institution',
       'archive_link', 'college_url', 'scholar_profile_url', 'State',
       'Specialisation', 'AreaofInterest', 'UGDegree', 'UGDepartment',
       'UGInstitute', 'PGDegree', 'PGDepartment', 'PGInstitute', 'MPhil',
       'MPhilDepartment', 'MPhilInstitute', 'Doctoral', 'DoctoralDepartment',
       'DoctoralInstitute', 'FirstJobPosition', 'FirstJobInstitution',
       'FirstJobDuration', 'SecondJobPosition', 'SecondJobInstitution',
       'SecondJobDuration', 'ThirdJobPosition', 'ThirdJobInstitution',
       'ThirdJobDuration', 'FourthJobPosition', 'FourthJobInstitution',
       'FourthJobDuration', 'FifthJobPosition', 'FifthJobInstitution',
       'FifthJobDuration', 'unique_id', 'university_type', 'name',
       'institution', 'UniversityType_id', 'State_id', 'Name_id',
       'Institution_id', 'Current_Designation', 'dept', 'state',
       'CurrentDesignation_r', 'archive_timeline', 'identif

In [ ]:
import pandas as pd

# Load datasets
master_dataset = pd.read_csv('/content/drive/My Drive/clean/clean/master_dataset.csv')
temporarycareer1 = pd.read_csv('/content/drive/My Drive/clean/clean/temporarycareer1.csv')

# Step 1: Identify columns in temporarycareer1 that are not in master_dataset
new_columns = [col for col in temporarycareer1.columns if col not in master_dataset.columns]
print("New columns to add from temporarycareer1:")
print(new_columns)

# Step 2: Count columns before merging
print(f"Columns in master_dataset before: {len(master_dataset.columns)}")

# Step 3: Add the new columns from temporarycareer1 to master_dataset
master_dataset[new_columns] = temporarycareer1[new_columns]

# Step 4: Count columns after merging
print(f"Columns in master_dataset after: {len(master_dataset.columns)}")

# Step 5: Save the updated master_dataset with new columns
master_dataset.to_csv('/content/drive/My Drive/clean/clean/master_dataset_updated.csv', index=False)

print(f"Updated master dataset saved with {len(new_columns)} new columns.")


New columns to add from temporarycareer1:
['email', 'description', 'headline', 'fullName', 'connectionsCount', 'FirstJobInstitution url', 'jobLocation', 'jobDateRange', 'jobStartedSince', 'jobIsCurrent', 'SecondJobInstitution url', 'jobLocation2', 'jobDateRange2', 'jobStartedSince2', 'jobIsCurrent2', 'school', 'schoolUrl', 'schoolDegree', 'schoolDescription', 'schoolDateRange', 'school2', 'schoolUrl2', 'schoolDegree2', 'schoolDateRange2', 'professionalEmail']
Columns in master_dataset before: 71
Columns in master_dataset after: 96
Updated master dataset saved with 25 new columns.


In [ ]:
import pandas as pd

# Load the datasets
master_dataset_updated = pd.read_csv('/content/drive/My Drive/clean/clean/master_dataset_updated.csv')
temporarycareer1 = pd.read_csv('/content/drive/My Drive/clean/clean/temporarycareer1.csv')

# Step 1: Copy columns from temporarycareer1 to master_dataset_updated based on 'Name'
columns_to_copy = [
    'email', 'description', 'headline', 'fullName', 'connectionsCount',
    'FirstJobInstitution', 'FirstJobInstitution url', 'FirstJobPosition',
    'jobLocation', 'jobDateRange', 'jobStartedSince', 'jobIsCurrent',
    'FirstJobDuration', 'SecondJobInstitution', 'SecondJobInstitution url',
    'SecondJobPosition', 'jobLocation2', 'jobDateRange2', 'jobStartedSince2',
    'jobIsCurrent2', 'SecondJobDuration', 'school', 'schoolUrl', 'schoolDegree',
    'schoolDescription', 'schoolDateRange', 'school2', 'schoolUrl2', 'schoolDegree2',
    'schoolDateRange2', 'professionalEmail'
]

# Iterate through each row of master_dataset_updated and copy matching info from temporarycareer1
for column in columns_to_copy:
    master_dataset_updated[column] = master_dataset_updated['Name'].map(
        dict(zip(temporarycareer1['Name'], temporarycareer1[column]))
    )

# Step 2: Save the final dataset with copied information
master_dataset_updated.to_csv('/content/drive/My Drive/clean/clean/final_master_dataset.csv', index=False)

print(f"Info copied from temporarycareer1 to master_dataset_updated based on 'Name' column.")


Info copied from temporarycareer1 to master_dataset_updated based on 'Name' column.


In [ ]:
# Count the number of rows
num_rows = master_dataset_updated.shape[0]

print(f"Number of rows in the filtered master dataset: {num_rows}")


Number of rows in the filtered master dataset: 2636
